# 06 — Lakeview Dashboard (AI/BI)

Creates a published Lakeview dashboard backed by the gold star schema —
parameterized by `SQL_WAREHOUSE_ID`, idempotent on re-run.

**6 datasets** × **9 widgets** laid out on a single page:

| Row | Widgets |
|-----|---------|
| Top  (0–3) | 4 KPI counters: Games · Shots · Goals · Shooting % |
| Mid  (3–9) | Standings bar chart  ·  Player season leaders table |
| Mid2 (9–15) | Shot-map heatmap (rink x/y)  ·  Events per game (line) |
| Bot (15–21) | Possession by team (pie) |

> **Schema note.** Lakeview JSON has two layers: `datasets` (the SQL queries) and `pages[].layout[]` (where each entry pairs a `widget` — name, query, viz spec — with a `position`: x, y, width, height on a 12-column grid). A dashboard with only datasets and no `layout` *creates* but renders blank; this notebook supplies both.

In [1]:
import os, json
from dotenv import load_dotenv
load_dotenv()

# Dual-mode Spark session — works locally via Databricks Connect AND inside a
# Databricks workspace notebook. In the workspace, `spark` already exists.
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [2]:
UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "sportlogiq_nhl")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

print(f"Catalog: {UC_CATALOG}")
print(f"Bronze:  {UC_CATALOG}.{BRONZE_SCHEMA}")
print(f"Silver:  {UC_CATALOG}.{SILVER_SCHEMA}")
print(f"Gold:    {UC_CATALOG}.{GOLD_SCHEMA}")
print(f"Volume:  {VOLUME_PATH}")

Catalog: alexander_booth
Bronze:  alexander_booth.sportlogiq_nhl_bronze
Silver:  alexander_booth.sportlogiq_nhl_silver
Gold:    alexander_booth.sportlogiq_nhl_gold
Volume:  /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data


In [3]:
from databricks.sdk.service.dashboards import Dashboard

G = f"{UC_CATALOG}.{GOLD_SCHEMA}"
SQL_WAREHOUSE_ID = os.getenv("SQL_WAREHOUSE_ID")
if not SQL_WAREHOUSE_ID:
    raise ValueError("SQL_WAREHOUSE_ID not set in .env")
print(f"Gold: {G}\nWarehouse: {SQL_WAREHOUSE_ID}")

Gold: alexander_booth.sportlogiq_nhl_gold
Warehouse: 49458fa9801b109b


In [4]:
DASHBOARD_NAME = "SportLogiq NHL — Hockey Analytics"

# ------------------------------------------------------------------ datasets
datasets = [
    {"name": "kpi_summary", "displayName": "League KPIs",
     "queryLines": [
         "SELECT\n",
         "  COUNT(DISTINCT game_id) AS games,\n",
         "  SUM(CASE WHEN is_shot THEN 1 ELSE 0 END) AS shots,\n",
         "  SUM(CASE WHEN is_goal THEN 1 ELSE 0 END) AS goals,\n",
         "  ROUND(100.0 * SUM(CASE WHEN is_goal THEN 1 ELSE 0 END)\n",
         "        / NULLIF(SUM(CASE WHEN is_shot THEN 1 ELSE 0 END), 0), 2) AS shooting_pct\n",
         f"FROM {G}.fact_game_events",
     ]},
    {"name": "team_standings", "displayName": "Standings",
     "queryLines": [
         f"SELECT team_shorthand, conference, division, wins, losses, overtime_losses, standings_points\n",
         f"FROM {G}.v_team_standings\n",
         "ORDER BY standings_points DESC",
     ]},
    {"name": "season_leaders", "displayName": "Player Season Leaders",
     "queryLines": [
         "SELECT full_name, team_shorthand, position, topic_id, metric_key, metric_value\n",
         f"FROM {G}.v_player_season_leaders\n",
         "ORDER BY metric_value DESC\n",
         "LIMIT 100",
     ]},
    {"name": "shot_map", "displayName": "Shot Map (x/y)",
     "queryLines": [
         "SELECT x_coord, y_coord, SUM(shot_attempts) AS shot_attempts, SUM(goals) AS goals\n",
         f"FROM {G}.v_shot_map\n",
         "WHERE x_coord IS NOT NULL AND y_coord IS NOT NULL\n",
         "GROUP BY x_coord, y_coord",
     ]},
    {"name": "events_per_game", "displayName": "Events per Game",
     "queryLines": [
         "SELECT game_date,\n",
         "       SUM(events) AS events,\n",
         "       SUM(shots)  AS shots,\n",
         "       SUM(goals)  AS goals\n",
         "FROM (\n",
         "  SELECT game_date,\n",
         "         COUNT(*) AS events,\n",
         "         SUM(CASE WHEN is_shot THEN 1 ELSE 0 END) AS shots,\n",
         "         SUM(CASE WHEN is_goal THEN 1 ELSE 0 END) AS goals\n",
         f"  FROM {G}.fact_game_events\n",
         "  WHERE game_date IS NOT NULL\n",
         "  GROUP BY game_date\n",
         ")\n",
         "GROUP BY game_date\n",
         "ORDER BY game_date",
     ]},
    {"name": "team_possession", "displayName": "Possession by Team",
     "queryLines": [
         "SELECT team, COUNT(*) AS plays_in_possession\n",
         f"FROM {G}.fact_game_events\n",
         "WHERE team IS NOT NULL\n",
         "GROUP BY team\n",
         "ORDER BY plays_in_possession DESC",
     ]},
]

# ------------------------------------------------------------------ widget builders
def counter(name, dataset, field, title, x, y, w_=3, h=3):
    return {
        "widget": {
            "name": name,
            "queries": [{"name": "main_query", "query": {
                "datasetName":  dataset,
                "fields":       [{"name": field, "expression": f"`{field}`"}],
                "disaggregated": True,
            }}],
            "spec": {
                "version": 2, "widgetType": "counter",
                "encodings": {"value": {"fieldName": field, "displayName": title}},
                "frame":     {"showTitle": True, "title": title},
            },
        },
        "position": {"x": x, "y": y, "width": w_, "height": h},
    }


def chart(name, dataset, widget_type, encodings, fields, x, y, w_, h, *, title=None,
          disaggregated=False):
    return {
        "widget": {
            "name": name,
            "queries": [{"name": "main_query", "query": {
                "datasetName":  dataset,
                "fields":       fields,
                "disaggregated": disaggregated,
            }}],
            "spec": {
                "version":    3,
                "widgetType": widget_type,
                "encodings":  encodings,
                "frame":      {"showTitle": True, "title": title or name},
            },
        },
        "position": {"x": x, "y": y, "width": w_, "height": h},
    }


def table(name, dataset, columns, x, y, w_, h, *, title=None):
    fields = [{"name": c["fieldName"], "expression": f"`{c['fieldName']}`"} for c in columns]
    return {
        "widget": {
            "name": name,
            "queries": [{"name": "main_query", "query": {
                "datasetName":  dataset,
                "fields":       fields,
                "disaggregated": True,
            }}],
            "spec": {
                "version":    1,
                "widgetType": "table",
                "encodings":  {"columns": columns},
                "frame":      {"showTitle": True, "title": title or name},
            },
        },
        "position": {"x": x, "y": y, "width": w_, "height": h},
    }


# ------------------------------------------------------------------ layout
layout = [
    # Row 0: 4 KPI counters
    counter("kpi_games",        "kpi_summary", "games",        "Games",        0, 0),
    counter("kpi_shots",        "kpi_summary", "shots",        "Shots",        3, 0),
    counter("kpi_goals",        "kpi_summary", "goals",        "Goals",        6, 0),
    counter("kpi_shooting_pct", "kpi_summary", "shooting_pct", "Shooting %",   9, 0),

    # Row 1: standings bar (left) + leaders table (right)
    chart(
        "standings_bar", "team_standings", "bar",
        encodings={
            "x": {"fieldName": "team_shorthand", "displayName": "Team",
                  "scale": {"type": "categorical"}},
            "y": {"fieldName": "standings_points", "displayName": "Points",
                  "scale": {"type": "quantitative"}},
            "color": {"fieldName": "conference", "displayName": "Conference",
                      "scale": {"type": "categorical"}},
        },
        fields=[
            {"name": "team_shorthand",   "expression": "`team_shorthand`"},
            {"name": "standings_points", "expression": "`standings_points`"},
            {"name": "conference",       "expression": "`conference`"},
        ],
        x=0, y=3, w_=6, h=6, title="Team Standings", disaggregated=True,
    ),
    table(
        "leaders_table", "season_leaders",
        columns=[
            {"fieldName": "full_name",       "displayName": "Player",  "type": "string"},
            {"fieldName": "team_shorthand",  "displayName": "Team",    "type": "string"},
            {"fieldName": "position",        "displayName": "Pos",     "type": "string"},
            {"fieldName": "metric_key",      "displayName": "Metric",  "type": "string"},
            {"fieldName": "metric_value",    "displayName": "Value",   "type": "float"},
        ],
        x=6, y=3, w_=6, h=6, title="Top Player-Season Metric Leaders",
    ),

    # Row 2: shot-map heatmap (left) + events line (right)
    chart(
        "shot_map_heatmap", "shot_map", "heatmap",
        encodings={
            "x": {"fieldName": "x_coord", "displayName": "Rink X",
                  "scale": {"type": "quantitative"}},
            "y": {"fieldName": "y_coord", "displayName": "Rink Y",
                  "scale": {"type": "quantitative"}},
            "color": {"fieldName": "shot_attempts", "displayName": "Attempts",
                      "aggregation": "sum", "scale": {"type": "quantitative"}},
        },
        fields=[
            {"name": "x_coord",       "expression": "`x_coord`"},
            {"name": "y_coord",       "expression": "`y_coord`"},
            {"name": "shot_attempts", "expression": "`shot_attempts`"},
        ],
        x=0, y=9, w_=6, h=6, title="Shot Map (rink coordinates)", disaggregated=True,
    ),
    chart(
        "events_line", "events_per_game", "line",
        encodings={
            "x": {"fieldName": "game_date", "displayName": "Game Date",
                  "scale": {"type": "temporal"}},
            "y": {"fieldName": "events", "displayName": "Events",
                  "scale": {"type": "quantitative"}},
        },
        fields=[
            {"name": "game_date", "expression": "`game_date`"},
            {"name": "events",    "expression": "`events`"},
        ],
        x=6, y=9, w_=6, h=6, title="Events per Game", disaggregated=True,
    ),

    # Row 3: possession by team (pie, full-width)
    chart(
        "possession_pie", "team_possession", "pie",
        encodings={
            "angle": {"fieldName": "plays_in_possession", "displayName": "Plays"},
            "color": {"fieldName": "team", "displayName": "Team",
                      "scale": {"type": "categorical"}},
        },
        fields=[
            {"name": "team",                "expression": "`team`"},
            {"name": "plays_in_possession", "expression": "`plays_in_possession`"},
        ],
        x=0, y=15, w_=12, h=6, title="Possession Plays by Team", disaggregated=True,
    ),
]

dashboard_json = {
    "datasets": datasets,
    "pages": [{
        "name":        "overview",
        "displayName": "SportLogiq Overview",
        "layout":      layout,
    }],
}
serialized = json.dumps(dashboard_json)
print(f"datasets: {len(datasets)} | widgets: {len(layout)} | serialized: {len(serialized):,} chars")

datasets: 6 | widgets: 9 | serialized: 7,784 chars


In [5]:
current_user = w.current_user.me()
parent_path  = f"/Workspace/Users/{current_user.user_name}"

# Try update-in-place if it already exists
existing_id = None
try:
    for obj in w.workspace.list(parent_path):
        if obj.path and obj.path.endswith(f"{DASHBOARD_NAME}.lvdash.json"):
            existing_id = str(obj.object_id)
            break
except Exception:
    pass

if existing_id:
    try:
        d = w.lakeview.get(dashboard_id=existing_id)
        w.lakeview.update(
            dashboard_id=d.dashboard_id,
            dashboard=Dashboard(serialized_dashboard=serialized, warehouse_id=SQL_WAREHOUSE_ID),
        )
        dashboard_id = d.dashboard_id
        print(f"Dashboard updated in place: {dashboard_id}")
    except Exception:
        existing_id = None

if not existing_id:
    d = w.lakeview.create(
        dashboard=Dashboard(
            display_name=DASHBOARD_NAME,
            warehouse_id=SQL_WAREHOUSE_ID,
            serialized_dashboard=serialized,
            parent_path=parent_path,
        )
    )
    dashboard_id = d.dashboard_id
    print(f"Dashboard created: {dashboard_id}")

w.lakeview.publish(dashboard_id=dashboard_id, warehouse_id=SQL_WAREHOUSE_ID, embed_credentials=True)
host = os.getenv("DATABRICKS_HOST", "").rstrip("/")
print(f"\nView at: {host}/sql/dashboardsv3/{dashboard_id}")

Dashboard created: 01f1457894d91c4d93ced2c6a77f8530



View at: https://e2-demo-field-eng.cloud.databricks.com/sql/dashboardsv3/01f1457894d91c4d93ced2c6a77f8530
